In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn
import xgboost
import shap

BASE = Path("/Users/ohaseelanka/disaster_analytics_hub")
FINAL = BASE / "data/final"
MODELS = BASE / "models"

MODELS.mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Flood training:", (FINAL / "flood_train.csv").exists())
print("Flood testing:", (FINAL / "flood_test.csv").exists())
print("Landslide training:", (FINAL / "landslide_train.csv").exists())
print("Landslide testing:", (FINAL / "landslide_test.csv").exists())

Python: /opt/anaconda3/envs/attendance/bin/python
Flood training: True
Flood testing: True
Landslide training: True
Landslide testing: True


/opt/anaconda3/envs/attendance/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

from pathlib import Path

BASE = Path.home() / "disaster_analytics_hub"

print("=== FLOOD DATASET AND MODEL FILES ===")

extensions = {".csv", ".joblib", ".pkl", ".json"}

matches = [
    p for p in BASE.rglob("*")
    if p.is_file()
    and p.suffix.lower() in extensions
    and any(word in p.name.lower() for word in [
        "flood", "train", "test", "split", "model"
    ])
    and ".venv" not in p.parts
    and "node_modules" not in p.parts
]

for p in sorted(matches):
    print(p.relative_to(BASE))

print(f"\nTotal matching files: {len(matches)}")


=== FLOOD DATASET AND MODEL FILES ===
data/backups/pp1_preprocessed/flood_human_impact_preprocessed_before_flag_fix.csv
data/backups/pp1_preprocessed/flood_human_impact_preprocessed_before_gis_fix.csv
data/backups/pp1_preprocessed/flood_human_impact_preprocessed_before_key_removal.csv
data/final/pp1_preprocessed/flood_ML_dataset_PP1_FINAL.csv
data/final/pp1_preprocessed/flood_human_impact_enriched_PP1.csv
data/interim/disaster_records/flood_event_ds_candidates.csv
data/interim/disaster_records/flood_event_ds_targets_provisional.csv
data/interim/disaster_records/flood_historical_clean.csv
data/interim/disaster_records/flood_recovered_impact_records.csv
data/processed/flood_impact_observations.csv
data/processed/flood_impact_with_all_demographics.csv
data/processed/flood_impact_with_ds_features.csv
data/processed/flood_impact_with_kalu_hydrology.csv
data/processed/flood_impact_with_kalu_kelani.csv
models/flood_RandomForest_real.joblib
models/flood_XGBoost_real.joblib
models/flood_rf_tune

In [4]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"

dataset_path = (
    BASE / "data/final/pp1_preprocessed"
    / "flood_ML_dataset_PP1_FINAL.csv"
)

df = pd.read_csv(dataset_path)

print("=== FLOOD DATASET ===")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nRisk-level distribution:")
print(df["risk_level"].value_counts(dropna=False))

print("\nMissing values:")
print(df.isna().sum().to_string())

print("\n=== TRAINING NOTEBOOKS ===")
for p in sorted((BASE / "notebooks").rglob("*.ipynb")):
    if any(word in p.name.lower() for word in ["human", "flood", "model"]):
        print(p.relative_to(BASE))

print("\n=== SAVED MODEL FILES ===")
for p in sorted((BASE / "models").glob("flood*.joblib")):
    print(p.name, f"({p.stat().st_size / 1024:.1f} KB)")


=== FLOOD DATASET ===
Shape: (802, 13)

Columns:
['District', 'Division', 'Date', 'episode_id', 'total_population', 'children_0_14_ratio', 'elderly_60_plus_ratio', 'female_ratio', 'unprotected_well_ratio', 'no_toilet_ratio', 'public_toilet_ratio', 'affected_target', 'risk_level']

Risk-level distribution:
risk_level
Moderate     207
Low          199
Very High    199
High         197
Name: count, dtype: int64

Missing values:
District                  0
Division                  0
Date                      0
episode_id                0
total_population          0
children_0_14_ratio       0
elderly_60_plus_ratio     0
female_ratio              0
unprotected_well_ratio    0
no_toilet_ratio           0
public_toilet_ratio       0
affected_target           0
risk_level                0

=== TRAINING NOTEBOOKS ===
notebooks/data/02_data_modeltraining.ipynb
notebooks/data/Model_Training/03_human_impact_model_training.ipynb

=== SAVED MODEL FILES ===
flood_RandomForest_real.joblib (1299.3 KB)

In [5]:

from pathlib import Path
import json

BASE = Path.home() / "disaster_analytics_hub"
notebook_path = (
    BASE / "notebooks/data/Model_Training"
    / "03_human_impact_model_training.ipynb"
)

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "train_test_split",
    "GroupShuffleSplit",
    "GroupKFold",
    "StratifiedGroupKFold",
    "RandomForestClassifier",
    "XGBClassifier",
    "GridSearchCV",
    "RandomizedSearchCV",
    "risk_level",
    "affected_target"
]

print("=== MODEL TRAINING CODE ===")

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(word in source for word in keywords):
        print(f"\n--- CELL {i} ---")
        print(source[:3500])


=== MODEL TRAINING CODE ===

--- CELL 0 ---

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier

BASE = Path("/Users/ohaseelanka/disaster_analytics_hub")
FINAL = BASE / "data/final"
MODELS = BASE / "models"
MODELS.mkdir(parents=True, exist_ok=True)

demographic_features = [
    "total_population",
    "children_0_14_ratio",
    "elderly_60_plus_ratio",
    "female_ratio",
    "unprotected_well_ratio",
    "no_toilet_ratio",
    "public_toilet_ratio"
]

landslide_features = demographic_features + [
    "pre_rfh_1dekad",
    "pre_r1h",
    "pre_r3h",
    "pre_rfq",
    "pre_r1q",
    "pre_r3q"
]

risk_names = ["Low", "Moderate", "High", "Very High"]
risk_to_number = {
    name: i for i, name in enumerate(risk_names)
}

results = []

fo

In [6]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"

print("=== ORIGINAL TRAIN / TEST FILES ===")

for hazard in ["flood", "landslide"]:
    for split in ["train", "test"]:
        path = BASE / f"data/final/{hazard}_{split}.csv"

        print(f"\n{hazard}_{split}.csv")
        print("Exists:", path.exists())

        if path.exists():
            df = pd.read_csv(path)
            print("Rows:", len(df))
            print("Columns:", len(df.columns))
            print("Unique episodes:", df["episode_id"].nunique())

print("\n=== AVAILABLE TRAINING CANDIDATES ===")

for path in sorted((BASE / "data/final").glob("*training_candidates.csv")):
    print(path.name)


=== ORIGINAL TRAIN / TEST FILES ===

flood_train.csv
Exists: False

flood_test.csv
Exists: False

landslide_train.csv
Exists: False

landslide_test.csv
Exists: False

=== AVAILABLE TRAINING CANDIDATES ===


In [7]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"
FOLDER = BASE / "data/final/pp1_processed"

print("=== PP1 TRAIN / TEST DATASETS ===")

for hazard in ["flood", "landslide"]:
    train_path = FOLDER / f"{hazard}_train.csv"
    test_path = FOLDER / f"{hazard}_test.csv"

    print(f"\n=== {hazard.upper()} ===")

    if not train_path.exists() or not test_path.exists():
        print("Files found in folder:")
        for p in sorted(FOLDER.glob("*")):
            print(" ", p.name)
        continue

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    print("Training records:", len(train))
    print("Testing records:", len(test))
    print("Training columns:", train.columns.tolist())

    if "episode_id" in train.columns:
        overlap = set(train["episode_id"].dropna()) & set(
            test["episode_id"].dropna()
        )
        print("Overlapping episode IDs:", len(overlap))

    print("Training risk distribution:")
    print(train["risk_level"].value_counts().to_string())

    print("Testing risk distribution:")
    print(test["risk_level"].value_counts().to_string())

print("\nNo files modified.")


=== PP1 TRAIN / TEST DATASETS ===

=== FLOOD ===
Files found in folder:

=== LANDSLIDE ===
Files found in folder:

No files modified.


In [8]:

from pathlib import Path

BASE = Path.home() / "disaster_analytics_hub"
FOLDER = BASE / "data/final"

print("=== ALL FILES INSIDE data/final ===")

for path in sorted(FOLDER.rglob("*")):
    if path.is_file():
        print(path.relative_to(BASE))

print("\n=== POSSIBLE TRAIN / TEST FILES ===")

matches = [
    p for p in BASE.rglob("*.csv")
    if any(word in p.name.lower() for word in [
        "train", "test", "split"
    ])
    and ".venv" not in p.parts
]

for path in sorted(matches):
    print(path.relative_to(BASE))

print("\nTotal possible files:", len(matches))
print("No files modified.")


=== ALL FILES INSIDE data/final ===
data/final/.DS_Store
data/final/pp1_preprocessed/.DS_Store
data/final/pp1_preprocessed/flood_ML_dataset_PP1_FINAL.csv
data/final/pp1_preprocessed/flood_human_impact_enriched_PP1.csv
data/final/pp1_preprocessed/flood_rf_tuning_results.csv
data/final/pp1_preprocessed/flood_test.csv
data/final/pp1_preprocessed/flood_train.csv
data/final/pp1_preprocessed/landslide_human_impact_preprocessed.csv

=== POSSIBLE TRAIN / TEST FILES ===
data/final/pp1_preprocessed/flood_test.csv
data/final/pp1_preprocessed/flood_train.csv

Total possible files: 2
No files modified.


In [9]:

from pathlib import Path
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

BASE = Path.home() / "disaster_analytics_hub"
FOLDER = BASE / "data/final/pp1_preprocessed"

train = pd.read_csv(FOLDER / "flood_train.csv")
test = pd.read_csv(FOLDER / "flood_test.csv")

features = [
    "total_population",
    "children_0_14_ratio",
    "elderly_60_plus_ratio",
    "female_ratio",
    "unprotected_well_ratio",
    "no_toilet_ratio",
    "public_toilet_ratio"
]

print("=== FLOOD DATASET VERIFICATION ===")
print("Training records:", len(train))
print("Testing records:", len(test))
print("Training episodes:", train["episode_id"].nunique())
print("Testing episodes:", test["episode_id"].nunique())

overlap = set(train["episode_id"].dropna()) & set(
    test["episode_id"].dropna()
)
print("Overlapping episodes:", len(overlap))

assert len(overlap) == 0, "Data leakage: shared episode IDs!"
assert train[features + ["risk_level"]].notna().all().all()
assert test[features + ["risk_level"]].notna().all().all()

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(train[features], train["risk_level"])
predictions = model.predict(test[features])

print("\n=== BASELINE MODEL RESULTS ===")
print("Accuracy:", round(accuracy_score(test["risk_level"], predictions), 4))
print("Balanced accuracy:", round(
    balanced_accuracy_score(test["risk_level"], predictions), 4
))
print("Macro F1:", round(
    f1_score(test["risk_level"], predictions, average="macro"), 4
))

print("\nNo files modified.")


=== FLOOD DATASET VERIFICATION ===
Training records: 652
Testing records: 150
Training episodes: 170
Testing episodes: 43
Overlapping episodes: 0

=== BASELINE MODEL RESULTS ===
Accuracy: 0.3667
Balanced accuracy: 0.3692
Macro F1: 0.3648

No files modified.


In [10]:

import pandas as pd
from pathlib import Path

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_validate

BASE = Path.home() / "disaster_analytics_hub"

train = pd.read_csv(
    BASE / "data/final/pp1_preprocessed/flood_train.csv"
)

features = [
    "total_population",
    "children_0_14_ratio",
    "elderly_60_plus_ratio",
    "female_ratio",
    "unprotected_well_ratio",
    "no_toilet_ratio",
    "public_toilet_ratio"
]

X = train[features]
y = train["risk_level"]
groups = train["episode_id"]

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ))
    ]),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200,
        max_leaf_nodes=15,
        learning_rate=0.05,
        random_state=42
    )
}

cv = GroupKFold(n_splits=5)
results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        groups=groups,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro"
        },
        n_jobs=1
    )

    results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Balanced Accuracy":
            scores["test_balanced_accuracy"].mean(),
        "CV Macro F1": scores["test_macro_f1"].mean()
    })

comparison = pd.DataFrame(results).sort_values(
    "CV Macro F1",
    ascending=False
)

print("=== FLOOD MODEL COMPARISON ===")
print(comparison.round(4).to_string(index=False))

print("\nBest candidate:", comparison.iloc[0]["Model"])
print("\nNo files or existing models modified.")


=== FLOOD MODEL COMPARISON ===
               Model  CV Accuracy  CV Balanced Accuracy  CV Macro F1
       Random Forest       0.4326                0.4358       0.4181
         Extra Trees       0.4248                0.4230       0.4153
HistGradientBoosting       0.4187                0.4232       0.4112
 Logistic Regression       0.4019                0.4014       0.3737

Best candidate: Random Forest

No files or existing models modified.


In [11]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_validate

BASE = Path.home() / "disaster_analytics_hub"

train = pd.read_csv(
    BASE / "data/final/pp1_preprocessed/flood_train.csv"
)

features = [
    "total_population",
    "children_0_14_ratio",
    "elderly_60_plus_ratio",
    "female_ratio",
    "unprotected_well_ratio",
    "no_toilet_ratio",
    "public_toilet_ratio"
]

risk_order = ["Low", "Moderate", "High", "Very High"]
risk_map = {name: i for i, name in enumerate(risk_order)}

X = train[features]
y = train["risk_level"].map(risk_map)
groups = train["episode_id"]

class OrdinalForest(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=200, max_depth=4,
                 min_samples_leaf=5, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    def fit(self, X, y):
        self.models_ = []
        self.classes_ = np.array([0, 1, 2, 3])

        for threshold in range(3):
            binary_y = (np.asarray(y) > threshold).astype(int)

            model = RandomForestClassifier(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                class_weight="balanced",
                random_state=self.random_state,
                n_jobs=-1
            )

            model.fit(X, binary_y)
            self.models_.append(model)

        return self

    def predict(self, X):
        probabilities = np.column_stack([
            model.predict_proba(X)[:, 1]
            for model in self.models_
        ])

        # Enforce non-increasing exceedance probabilities.
        probabilities = np.minimum.accumulate(
            probabilities, axis=1
        )

        class_probabilities = np.column_stack([
            1 - probabilities[:, 0],
            probabilities[:, 0] - probabilities[:, 1],
            probabilities[:, 1] - probabilities[:, 2],
            probabilities[:, 2]
        ])

        return class_probabilities.argmax(axis=1)

model = OrdinalForest()

scores = cross_validate(
    model,
    X,
    y,
    groups=groups,
    cv=GroupKFold(n_splits=5),
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "macro_f1": "f1_macro"
    },
    n_jobs=1
)

print("=== ORDINAL FOREST — FLOOD ===")
print("CV Accuracy:", round(scores["test_accuracy"].mean(), 4))
print("CV Balanced Accuracy:",
      round(scores["test_balanced_accuracy"].mean(), 4))
print("CV Macro F1:", round(scores["test_macro_f1"].mean(), 4))

print("\nRandom Forest previous CV Macro F1: 0.4181")
print("No files or saved models modified.")


=== ORDINAL FOREST — FLOOD ===
CV Accuracy: 0.411
CV Balanced Accuracy: 0.4122
CV Macro F1: 0.3152

Random Forest previous CV Macro F1: 0.4181
No files or saved models modified.


In [12]:

import pandas as pd
from pathlib import Path

BASE = Path.home() / "disaster_analytics_hub"

train = pd.read_csv(
    BASE / "data/final/pp1_preprocessed/flood_train.csv"
)

features = [
    "total_population",
    "children_0_14_ratio",
    "elderly_60_plus_ratio",
    "female_ratio",
    "unprotected_well_ratio",
    "no_toilet_ratio",
    "public_toilet_ratio"
]

print("=== FEATURE INFORMATION AUDIT ===")

print("\nUnique values per feature:")
print(train[features].nunique().to_string())

print("\n=== IDENTICAL FEATURES, DIFFERENT RISK LEVELS ===")

feature_groups = train.groupby(features, dropna=False).agg(
    records=("risk_level", "size"),
    distinct_risk_levels=("risk_level", "nunique"),
    distinct_episodes=("episode_id", "nunique")
).reset_index()

conflicting = feature_groups[
    feature_groups["distinct_risk_levels"] > 1
]

print("Unique feature combinations:", len(feature_groups))
print("Combinations with conflicting risk labels:", len(conflicting))
print(
    "Records in conflicting combinations:",
    int(conflicting["records"].sum())
)

print("\n=== DS DIVISION RISK VARIATION ===")

ds_groups = train.groupby(
    ["District", "Division"]
).agg(
    records=("risk_level", "size"),
    distinct_risk_levels=("risk_level", "nunique"),
    distinct_episodes=("episode_id", "nunique")
)

varying = ds_groups[
    ds_groups["distinct_risk_levels"] > 1
]

print("DS divisions:", len(ds_groups))
print("DS divisions with multiple risk levels:", len(varying))

print("\nExamples:")
print(
    varying.sort_values(
        "distinct_risk_levels", ascending=False
    ).head(10).to_string()
)

print("\nNo files modified.")


=== FEATURE INFORMATION AUDIT ===

Unique values per feature:
total_population          68
children_0_14_ratio       68
elderly_60_plus_ratio     68
female_ratio              68
unprotected_well_ratio    68
no_toilet_ratio           68
public_toilet_ratio       61

=== IDENTICAL FEATURES, DIFFERENT RISK LEVELS ===
Unique feature combinations: 68
Combinations with conflicting risk labels: 65
Records in conflicting combinations: 643

=== DS DIVISION RISK VARIATION ===
DS divisions: 68
DS divisions with multiple risk levels: 65

Examples:
                         records  distinct_risk_levels  distinct_episodes
District  Division                                                       
Ratnapura Ratnapura           14                     4                 12
Gampaha   Gampaha             26                     4                 22
Kalutara  Palindanuwara        6                     4                  6
          Mathugama            6                     4                  6
          Madu

In [13]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"

files = [
    BASE / "data/final/pp1_preprocessed/flood_human_impact_enriched_PP1.csv",
    BASE / "data/processed/flood_impact_with_kalu_hydrology.csv",
    BASE / "data/processed/flood_impact_with_kalu_kelani.csv",
    BASE / "data/processed/flood_impact_observations.csv"
]

keywords = [
    "rain", "river", "water", "level", "flood",
    "depth", "severity", "extent", "discharge",
    "flow", "duration", "elevation"
]

for path in files:
    print(f"\n=== {path.name} ===")

    if not path.exists():
        print("File not found")
        continue

    df = pd.read_csv(path)
    print("Total records:", len(df))

    matching_columns = [
        col for col in df.columns
        if any(word in col.lower() for word in keywords)
    ]

    if not matching_columns:
        print("No hazard-related columns found")
        continue

    report = pd.DataFrame({
        "Column": matching_columns,
        "Non-missing": [
            df[col].notna().sum()
            for col in matching_columns
        ],
        "Unique values": [
            df[col].nunique(dropna=True)
            for col in matching_columns
        ]
    })

    print(report.to_string(index=False))

print("\nNo files modified.")



=== flood_human_impact_enriched_PP1.csv ===
Total records: 802
    Column  Non-missing  Unique values
risk_level          802              4

=== flood_impact_with_kalu_hydrology.csv ===
Total records: 877
No hazard-related columns found

=== flood_impact_with_kalu_kelani.csv ===
Total records: 877
        Column  Non-missing  Unique values
kelani_rain_1d           17             17
kelani_rain_3d           17             17
kelani_rain_7d           17             17

=== flood_impact_observations.csv ===
Total records: 877
No hazard-related columns found

No files modified.


In [14]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"

files = [
    "data/processed/flood_impact_with_kalu_hydrology.csv",
    "data/processed/flood_impact_with_kalu_kelani.csv"
]

for relative_path in files:
    path = BASE / relative_path
    df = pd.read_csv(path)

    print(f"\n=== {path.name} ===")
    print("Records:", len(df))

    summary = pd.DataFrame({
        "Column": df.columns,
        "Data type": df.dtypes.astype(str).values,
        "Available": df.notna().sum().values,
        "Missing": df.isna().sum().values,
        "Unique": df.nunique(dropna=True).values
    })

    print(summary.to_string(index=False))

print("\nNo files modified.")



=== flood_impact_with_kalu_hydrology.csv ===
Records: 877
                 Column Data type  Available  Missing  Unique
               District       str        877        0       5
               Division       str        877        0      68
                   Date       str        877        0     278
    source_record_count     int64        877        0      18
           affected_min   float64        877        0     567
           affected_max   float64        877        0     588
affected_sum_unverified   float64        877        0     602
          source_sheets       str        877        0      20
          target_status       str        877        0       2
        affected_target   float64        802       75     552
         episode_number     int64        877        0      94
             episode_id       str        877        0     239
          _district_key       str        877        0       5
                _ds_key       str        877        0      68
          d

In [15]:

from pathlib import Path
import pandas as pd

BASE = Path.home() / "disaster_analytics_hub"
FOLDER = BASE / "data/final/pp1_preprocessed"

train = pd.read_csv(FOLDER / "flood_train.csv")
test = pd.read_csv(FOLDER / "flood_test.csv")

hydro = pd.read_csv(
    BASE / "data/processed/flood_impact_with_kalu_kelani.csv"
)

keys = ["District", "Division", "Date", "episode_id"]

hydro_columns = [
    "pre_wl_1d",
    "pre_wl_max_3d",
    "pre_wl_avg_7d",
    "pre_q_1d",
    "pre_q_max_3d",
    "pre_q_avg_7d",
    "kelani_rain_1d",
    "kelani_wl_1d",
    "kelani_q_1d"
]

print("=== MATCH KEY CHECK ===")
print("Duplicate hydrology keys:",
      hydro.duplicated(keys).sum())

if hydro.duplicated(keys).any():
    print("Duplicate keys must be reviewed before merging.")
else:
    lookup = hydro[keys + hydro_columns].copy()

    for name, split in [
        ("TRAIN", train),
        ("TEST", test)
    ]:
        merged = split[keys].merge(
            lookup,
            on=keys,
            how="left",
            validate="many_to_one"
        )

        kalu_complete = merged[
            hydro_columns[:6]
        ].notna().all(axis=1)

        kelani_complete = merged[
            hydro_columns[6:]
        ].notna().all(axis=1)

        print(f"\n=== {name} ===")
        print("Total records:", len(split))
        print("Kalu hydrology complete:",
              int(kalu_complete.sum()))
        print("Kelani measurements complete:",
              int(kelani_complete.sum()))
        print("Any hydrology available:",
              int((kalu_complete | kelani_complete).sum()))

print("\nNo files modified.")


=== MATCH KEY CHECK ===
Duplicate hydrology keys: 0

=== TRAIN ===
Total records: 652
Kalu hydrology complete: 92
Kelani measurements complete: 15
Any hydrology available: 107

=== TEST ===
Total records: 150
Kalu hydrology complete: 29
Kelani measurements complete: 2
Any hydrology available: 31

No files modified.


In [16]:

from pathlib import Path

BASE = Path.home() / "disaster_analytics_hub"

print("=== EXISTING BACKEND FILES ===")

for folder_name in ["backend", "api", "src"]:
    folder = BASE / folder_name

    print(f"\n{folder_name}/")

    if not folder.exists():
        print("Folder does not exist")
        continue

    for path in sorted(folder.rglob("*.py")):
        if any(part in {
            ".venv", "__pycache__", "site-packages"
        } for part in path.parts):
            continue

        print(" ", path.relative_to(BASE))

print("\n=== FASTAPI INSTALLATION ===")

try:
    import fastapi
    import uvicorn

    print("FastAPI:", fastapi.__version__)
    print("Uvicorn:", uvicorn.__version__)
except ImportError as error:
    print("Missing package:", error)

print("\nNo files modified.")


=== EXISTING BACKEND FILES ===

backend/
Folder does not exist

api/
  api/__init__.py
  api/main.py
  api/main_before_dashboard_ml.py
  api/main_before_ml_integration.py

src/
  src/__init__.py
  src/decision_recommendations.py
  src/decision_support_engine.py
  src/ds_demographic_lookup.py
  src/ds_hazard_inputs.py
  src/evaluation/__init__.py
  src/features/__init__.py
  src/human_exposure_assessment.py
  src/human_impact_predictor.py
  src/human_impact_scoring.py
  src/infrastructure_risk.py
  src/ingestion/__init__.py
  src/models/__init__.py
  src/multi_hazard_integration.py
  src/multi_hazard_scoring.py
  src/preprocessing/__init__.py
  src/spatial/__init__.py
  src/targets/__init__.py
  src/utils/__init__.py

=== FASTAPI INSTALLATION ===
FastAPI: 0.143.1
Uvicorn: 0.54.0

No files modified.


In [17]:

from pathlib import Path
import ast

BASE = Path.home() / "disaster_analytics_hub"

files = [
    BASE / "api/main.py",
    BASE / "src/human_impact_predictor.py",
    BASE / "src/decision_support_engine.py",
    BASE / "src/multi_hazard_integration.py"
]

for path in files:
    print("\n" + "=" * 75)
    print(path.relative_to(BASE))
    print("=" * 75)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    print("Total lines:", len(lines))

    try:
        tree = ast.parse(source)

        print("\nImports:")
        for node in tree.body:
            if isinstance(node, ast.Import):
                print(" ", ", ".join(
                    alias.name for alias in node.names
                ))
            elif isinstance(node, ast.ImportFrom):
                print(" ", node.module)

        print("\nFunctions and classes:")
        for node in tree.body:
            if isinstance(
                node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
            ):
                print(f"  Line {node.lineno}: {node.name}")

        if path.name == "main.py":
            print("\nAPI route declarations:")
            for i, line in enumerate(lines, start=1):
                if any(route in line for route in [
                    "@app.get", "@app.post",
                    "@app.put", "@app.delete",
                    "@router.get", "@router.post"
                ]):
                    print(f"  Line {i}: {line.strip()}")

    except SyntaxError as error:
        print("Syntax error:", error)

print("\nNo files modified.")



api/main.py
Total lines: 309

Imports:
  functools
  fastapi
  fastapi.middleware.cors
  fastapi.encoders
  pydantic
  src.decision_support_engine
  src.decision_recommendations
  src.ds_demographic_lookup
  src.infrastructure_risk
  src.human_exposure_assessment
  src.human_impact_predictor

Functions and classes:
  Line 39: root
  Line 48: health
  Line 57: AnalysisRequest
  Line 80: get_engine
  Line 85: get_demographics
  Line 90: get_example_infrastructure
  Line 124: analyze
  Line 257: MLPredictionRequest
  Line 269: get_ml_predictor
  Line 274: predict_human_impact

API route declarations:
  Line 38: @app.get("/")
  Line 47: @app.get("/api/health")
  Line 123: @app.post("/api/analyze")
  Line 273: @app.post("/api/ml/predict")

src/human_impact_predictor.py
Total lines: 122

Imports:
  pathlib
  joblib
  numpy
  pandas
  shap

Functions and classes:
  Line 27: HumanImpactPredictor

src/decision_support_engine.py
Total lines: 269

Imports:
  src.multi_hazard_integration
  src.mu